# 🧪 NutriTrack — Complete Test Suite

Runs all test scripts directly by importing functions from the `app/tests` directory.
- Output metrics are gathered into specific DataFrames.
- Raw data responses (p1-p4, q1-q6) are saved to `app/data/tests` for deep inspection.
- API Server is briefly spawned inside the notebook to run API endpoint tests.

## 0. Setup & Imports

In [1]:
import sys, os, time, json
import pandas as pd
from IPython.display import display

# Ensure project root is in path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, "config", ".env"))

import subprocess
import requests
import time
import tests.test_api as api_tests

FAST_FOOD_IMG = os.path.join(project_root, "..", "data", "images", "food", "fast_food.jpg")
COM_TAM_IMG   = os.path.join(project_root, "..", "data", "images", "food", "com_tam.jpg")

TEST_OUTPUT_DIR = os.path.join(project_root, "data", "tests")
os.makedirs(TEST_OUTPUT_DIR, exist_ok=True)

def save_raw_output(filename: str, data: dict|list):
    if not data: return
    path = os.path.join(TEST_OUTPUT_DIR, filename)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"Saved {filename}")

print(f"✅ Setup complete. Outputs will be saved to: {TEST_OUTPUT_DIR}")

✅ Setup complete. Outputs will be saved to: d:\Project\Code\nutritrack-documentation\app\data\tests


## 1. USDA Client Tests

In [2]:
from third_apis.USDA import USDAClient
import tests.test_usda_client as usda_tests

usda_client = USDAClient(api_key=os.getenv("USDA_API_KEY"))

2026-03-06 05:45:39 | INFO     | third_apis.USDA                | USDAClient initialized (api_key=***)


In [3]:
r1 = usda_tests.test_normalize_query(usda_client)
r2 = usda_tests.test_get_PCF_mock(usda_client)
r3 = usda_tests.test_get_PCF_real(usda_client)
r4 = usda_tests.test_get_ingredients_mock(usda_client)
r5 = usda_tests.test_get_ingredients_real(usda_client)
r6 = usda_tests.test_search_best_cache(usda_client)
r7 = usda_tests.test_get_PCF_and_ingredients_mock(usda_client)
r8 = usda_tests.test_get_PCF_and_ingredients_real(usda_client)

usda_results = [r1, r2, r3, r4, r5, r6, r7, r8]
print(f"\nUSDA Client tests passed: {sum(1 for r in usda_results if r)}/{len(usda_results)}")

                                                                                                    
                           ╔════════════════════════════════════════════╗                           
                           ║      Starting test: _normalize_query       ║                           
                           ╚════════════════════════════════════════════╝                           
2026-03-06 05:24:21 | INFO     | tests.test_usda_client         | ✅ PASS | 'rice (gạo)' → 'rice'
2026-03-06 05:24:21 | INFO     | tests.test_usda_client         | ✅ PASS | 'bò (beef)' → 'bo'
2026-03-06 05:24:21 | INFO     | tests.test_usda_client         | ✅ PASS | 'gà (chicken)' → 'ga'
2026-03-06 05:24:21 | INFO     | tests.test_usda_client         | ✅ PASS | 'cá (fish)' → 'ca'
2026-03-06 05:24:21 | INFO     | tests.test_usda_client         | ✅ PASS | 'crème brûlée (dessert)' → 'creme brulee'
2026-03-06 05:24:21 | INFO     | tests.test_usda_client         | ✅ PASS | 'pâté (duck)' → 'pate'

## 2. Qwen3VL Model Tests
Runs the 3 model inference methods directly against the test images and records metrics.

In [4]:
from models.QWEN3VL import Qwen3VL
import tests.test_qwen_client as qwen_tests

qwen = Qwen3VL()

                                                                                                    
                           ╔════════════════════════════════════════════╗                           
                           ║            Initializing Qwen3VL            ║                           
                           ╚════════════════════════════════════════════╝                           
2026-03-06 05:24:42 | INFO     | models.QWEN3VL                 | model=qwen.qwen3-vl-235b-a22b, region=us-east-1
2026-03-06 05:24:42 | INFO     | models.QWEN3VL                 | Qwen3VL ready! (model=qwen.qwen3-vl-235b-a22b, region=us-east-1)


In [5]:
# ─── 1 Dish Image (com_tam.jpg) ───
q1 = qwen_tests.test_analyze_food(qwen, COM_TAM_IMG, expected_min=1, label="com_tam")
q1['method'] = 'Method 1 (Converse)'
q1['image'] = 'com_tam'
save_raw_output("q1_method1_com_tam.json", q1.get("raw_output"))

q2 = qwen_tests.test_analyze_food_with_instructor(qwen, COM_TAM_IMG, expected_min=1, label="com_tam")
q2['method'] = 'Method 2 (Instructor)'
q2['image'] = 'com_tam'
save_raw_output("q2_method2_com_tam.json", q2.get("raw_output"))

q3 = qwen_tests.test_analyze_food_with_tools(qwen, COM_TAM_IMG, expected_min=1, label="com_tam")
q3['method'] = 'Method 3 (Tools)'
q3['image'] = 'com_tam'
save_raw_output("q3_method3_com_tam.json", q3.get("raw_output"))

# ─── Multi-Dish Image (fast_food.jpg) ───
q4 = qwen_tests.test_analyze_food(qwen, FAST_FOOD_IMG, expected_min=10, label="fast_food")
q4['method'] = 'Method 1 (Converse)'
q4['image'] = 'fast_food'
save_raw_output("q4_method1_fast_food.json", q4.get("raw_output"))

q5 = qwen_tests.test_analyze_food_with_instructor(qwen, FAST_FOOD_IMG, expected_min=10, label="fast_food")
q5['method'] = 'Method 2 (Instructor)'
q5['image'] = 'fast_food'
save_raw_output("q5_method2_fast_food.json", q5.get("raw_output"))

q6 = qwen_tests.test_analyze_food_with_tools(qwen, FAST_FOOD_IMG, expected_min=10, label="fast_food")
q6['method'] = 'Method 3 (Tools)'
q6['image'] = 'fast_food'
save_raw_output("q6_method3_fast_food.json", q6.get("raw_output"))


                                                                                                    
                           ╔════════════════════════════════════════════╗                           
                           ║      Method 1: analyze_food [com_tam]      ║                           
                           ╚════════════════════════════════════════════╝                           
2026-03-06 05:24:44 | INFO     | models.QWEN3VL                 | analyze_food() called for image: d:\Project\Code\nutritrack-documentation\app\..\data\images\food\com_tam.jpg
2026-03-06 05:24:44 | INFO     | models.QWEN3VL                 | [Converse] Loading image: d:\Project\Code\nutritrack-documentation\app\..\data\images\food\com_tam.jpg
2026-03-06 05:24:44 | INFO     | utils.processor                | Preparing image for Bedrock: d:\Project\Code\nutritrack-documentation\app\..\data\images\food\com_tam.jpg
2026-03-06 05:24:44 | INFO     | models.QWEN3VL                 | [Converse] An

In [6]:
qwen_results = [q1, q2, q3, q4, q5, q6]

print(f"\nQwen3VL tests passed: {sum(1 for r in qwen_results if r.get('success', False))}/{len(qwen_results)}")
print("(Note: Instructor tests are expected to fail with bedrock JSON schema)")

df_qwen = pd.DataFrame(qwen_results)
cols = ['method', 'image', 'status', 'time_s', 'dishes', 'ingredients', 'bedrock_calls', 'usda_calls', 'usda_cache_hits', 'tool_rounds', 'notes']
cols = [c for c in cols if c in df_qwen.columns]

display(df_qwen[cols].fillna(''))


Qwen3VL tests passed: 4/6
(Note: Instructor tests are expected to fail with bedrock JSON schema)


,method,image,status,time_s,dishes,ingredients,bedrock_calls,usda_calls,usda_cache_hits,tool_rounds,notes
0,Method 1 (Converse),com_tam,✅ PASS,12.5,1,11,1,0,0,0,
1,Method 2 (Instructor),com_tam,⚠️ KNOWN,3.5,0,0,1,0,0,0,BEDROCK_JSON limit / Empty results
2,Method 3 (Tools),com_tam,✅ PASS,24.3,1,8,3,7,3,3,L2 disk hits: 3
3,Method 1 (Converse),fast_food,✅ PASS,48.2,12,42,1,0,0,0,
4,Method 2 (Instructor),fast_food,⚠️ KNOWN,4.0,0,0,1,0,0,0,BEDROCK_JSON limit / Empty results
5,Method 3 (Tools),fast_food,✅ PASS,89.5,10,35,3,26,1,3,L2 disk hits: 1


## 3. Pipeline Tests
Testing the high-level orchestrated pipeline flows.

In [7]:
import tests.test_pipeline as pipeline_tests

# ─── 1 Dish Image (com_tam.jpg) ───
p1 = pipeline_tests.test_pipeline_tools(qwen, usda_client, COM_TAM_IMG, expected_min=1, label="com_tam")
p1['method'] = 'Pipeline (Tools)'
p1['image'] = 'com_tam'
save_raw_output("p1_pipeline_tools_com_tam.json", p1.get("raw_output"))

p2 = pipeline_tests.test_pipeline_manual(qwen, usda_client, COM_TAM_IMG, expected_min=1, label="com_tam")
p2['method'] = 'Pipeline (Manual)'
p2['image'] = 'com_tam'
save_raw_output("p2_pipeline_manual_com_tam.json", p2.get("raw_output"))

# ─── Multi-Dish Image (fast_food.jpg) ───
p3 = pipeline_tests.test_pipeline_tools(qwen, usda_client, FAST_FOOD_IMG, expected_min=10, label="fast_food")
p3['method'] = 'Pipeline (Tools)'
p3['image'] = 'fast_food'
save_raw_output("p3_pipeline_tools_fast_food.json", p3.get("raw_output"))

p4 = pipeline_tests.test_pipeline_manual(qwen, usda_client, FAST_FOOD_IMG, expected_min=10, label="fast_food")
p4['method'] = 'Pipeline (Manual)'
p4['image'] = 'fast_food'
save_raw_output("p4_pipeline_manual_fast_food.json", p4.get("raw_output"))


                                                                                                    
                          ╔═════════════════════════════════════════════╗                           
                          ║   Pipeline Test: method='tools' [com_tam]   ║                           
                          ╚═════════════════════════════════════════════╝                           
2026-03-06 05:27:47 | INFO     | third_apis.USDA                | L1 RAM cache cleared
                                                                                                    
                           ╔════════════════════════════════════════════╗                           
                           ║        Nutrition Analysis Pipeline         ║                           
                           ╚════════════════════════════════════════════╝                           
2026-03-06 05:27:47 | INFO     | scripts.pipeline               | Image: d:\Project\Code\nutritrack-docum

KeyboardInterrupt: 

In [ ]:
pipe_results = [p1, p2, p3, p4]
print(f"\nPipeline tests passed: {sum(1 for r in pipe_results if r.get('success', False))}/{len(pipe_results)}")

df_pipe = pd.DataFrame(pipe_results)
pipe_cols = ['method', 'image', 'status', 'time_s', 'dishes', 'ingredients', 'bedrock_calls', 'usda_calls', 'usda_cache_hits', 'tool_rounds', 'notes']
pipe_cols = [c for c in pipe_cols if c in df_pipe.columns]

display(df_pipe[pipe_cols].fillna(''))


Pipeline tests passed: 4/4


,method,image,status,time_s,dishes,ingredients,bedrock_calls,usda_calls,usda_cache_hits,tool_rounds,notes
0,Pipeline (Tools),com_tam,✅ PASS,14.1,1,9,3,1,7,3,
1,Pipeline (Manual),com_tam,✅ PASS,23.2,1,12,1,13,11,0,
2,Pipeline (Tools),fast_food,✅ PASS,79.0,11,33,4,10,34,4,
3,Pipeline (Manual),fast_food,✅ PASS,83.6,12,46,1,1,12,0,


## 4. API Endpoint Tests
Automatically spawn the Uvicorn server, hit the endpoints, and close the server.

In [5]:
# Start fastapi server in background
print("Starting FASTAPI server in background...")
env = os.environ.copy()
server_process = subprocess.Popen(
    [sys.executable, "templates/api.py"], 
    cwd=project_root, 
    stdout=subprocess.PIPE, 
    stderr=subprocess.PIPE,
    env=env
)

Starting FASTAPI server in background...


In [ ]:
try:
    # Try hit healthcheck to confirm up
    resp = requests.get("http://localhost:8000/health", timeout=5)
    print("Server is UP.")
    
    a1 = api_tests.test_health_check()
    a2 = api_tests.test_analyze_invalid_file()
    a3 = api_tests.test_analyze_tools(COM_TAM_IMG, label="com_tam_tools", expected_min=1)
    a4 = api_tests.test_analyze_manual(COM_TAM_IMG, label="com_tam_manual", expected_min=1)

    api_results = [a1, a2, a3, a4]
    print(f"\nAPI tests passed: {sum(1 for r in api_results if r)}/{len(api_results)}")
    
except Exception as e:
    print(f"Failed during API tests: {e}")
finally:
    # Teardown server
    try:
        print("Terminating server...")
        server_process.terminate()
        server_process.wait()
        print("Server terminated.")
    except: pass

Server is UP.
                                                                                                    
                           ╔════════════════════════════════════════════╗                           
                           ║           API Test: GET /health            ║                           
                           ╚════════════════════════════════════════════╝                           
2026-03-06 05:00:11 | INFO     | tests.test_api                 | ✅ PASS | /health: {'status': 'ok', 'qwen_model': 'qwen.qwen3-vl-235b-a22b', 'usda_ready': True}
                                                                                                    
                         ╔═══════════════════════════════════════════════╗                          
                         ║   API Test: POST /analyze with invalid file   ║                          
                         ╚═══════════════════════════════════════════════╝                          
2026-03-06 05:00